# 04 — 非线性隐变量：直接数值积分
唯一结构性修改：将 $wx+b$ 换成神经网络 $f_\theta(x)$。先验和观测噪声不变。联合密度依旧容易逐点计算，但边缘密度与后验一般无解析解。由于隐变量只有一维，仍可用网格近似积分，并直接最大化观测似然。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 定义模型与计算边缘密度
沿用第 03 章的标准正态先验与固定二维观测噪声，仅将均值函数替换为神经网络。无法再直接使用 $ww^T+R$ 构造边缘高斯；回到第 03 章的网格积分方法。为使训练可以求导，使用 PyTorch，并在 log 空间计算。

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float32)
sigma = .45
# 解码器输入一个标量 x，输出二维条件均值 f_theta(x)。
decoder = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,2))
# 241 个隐变量取值，近似积分区间 [-6,6]。
grid = torch.linspace(-6,6,241).reshape(-1,1)

# 对每个观测，枚举隐变量网格。每行是一组联合对数密度。
def log_joint(obs, grid):
    # 每个网格点都有一个条件高斯分布，协方差固定。
    means = decoder(grid)
    # prior.log_prob 计算先验对数密度。
    lp = torch.distributions.Normal(0., 1.).log_prob(grid[:, 0])
    rows = []
    for i in range(len(obs)):
        # 同一个观测在所有网格点对应的二维条件分布下的密度。
        # 将观测复制到每个网格点。
        repeated_obs = obs[i].repeat(len(grid), 1)
        lc = torch.distributions.Normal(means, sigma).log_prob(repeated_obs).sum(dim=1)
        # log p(x,y) = log p(y|x) + log p(x)
        rows.append(lc + lp)
    return torch.stack(rows)

def grid_log_marginal(obs, grid):
    joint = log_joint(obs, grid)
    dx = grid[1, 0] - grid[0, 0]
    values = []
    for i in range(len(obs)):
        # log p(y_i) ≈ log(sum_j exp(log p(x_j,y_i)) * dx)
        values.append(torch.logsumexp(joint[i], dim=0) + torch.log(dx))
    return torch.stack(values)

# 与第 02、03 章相同：直接最小化负观测对数似然。
opt = torch.optim.Adam(decoder.parameters(), lr=.01)
history = []
for _ in range(350):
    # 每轮随机取 128 个观测来估计平均训练目标。
    batch = Y[torch.randint(len(Y), (128,))]
    ll = grid_log_marginal(batch, grid).mean()
    opt.zero_grad()
    (-ll).backward()
    opt.step()
    history.append(ll.item())
plot_history(history, 'mean log likelihood (grid)')
# 可选后验验证：不参与梯度 MLE。
with torch.no_grad():
    lj = log_joint(Y[:1], grid)[0]
    log_z = grid_log_marginal(Y[:1], grid)[0]
    posterior = []
    for value in lj:
        posterior.append(torch.exp(value-log_z).item())
    plt.plot(grid[:,0], posterior)
    plt.title('Posterior density for first observation')
    plt.show()
# 沿用前两章的先验采样、条件采样。
with torch.no_grad():
    samples = []
    for i in range(len(y)):
        x = torch.randn(1)
        samples.append((decoder(x) + sigma*torch.randn(2)).numpy())
    compare(y, np.array(samples))

In [ ]:
# 改变网格密度，观察同一组参数下的边缘对数密度是否趋于稳定。
with torch.no_grad():
    for size in [61,121,241,481,961]:
        g = torch.linspace(-6,6,size).reshape(-1,1)
        values = grid_log_marginal(Y[:10], g)
        print(size, values.mean().item())
    # 保存训练好的生成模型，第 05 章直接复用。
    torch.save(decoder.state_dict(), 'decoder_grid.pt')

## 小结
每维 241 个网格点，二维就需要 241² 个点，十维需要 241¹⁰ 个点。直接 MLE 的数学目标没有改变，但数值积分可能不再可行。第 04.5 章进一步区分逐点密度求值和边缘积分，并研究先验采样能否替代网格积分。